Imports

In [1]:
!pip -q install spacy
import re
import pandas as pd
import numpy as np
from pathlib import Path

import spacy
nlp = spacy.blank("en")

Config

In [2]:
from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/CS685/linkedin"
IN_PATH  = f"{BASE_DIR}/sentences_annotated_clean.csv"
OUT_DIR  = Path(f"{BASE_DIR}/ner_data_v3")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
TEST_FRAC = 0.15
DEV_FRAC  = 0.15

Mounted at /content/drive


Load and filter annotated rows

In [3]:
df = pd.read_csv(IN_PATH)

# normalize has_skill
def parse_bool(x):
    if pd.isna(x):
        return None
    s = str(x).strip().lower()
    if s in ["true", "t", "1", "yes"]: return True
    if s in ["false", "f", "0", "no"]: return False
    return None

df["has_skill_bool"] = df["has_skill"].apply(parse_bool)
df = df[df["has_skill_bool"].notna()].copy()
df = df.reset_index(drop=True)

print("Annotated rows:", len(df))
print(df[["domain","has_skill_bool"]].value_counts().head())

Annotated rows: 700
domain  has_skill_bool
SWE     True              268
        False             192
DA      False              87
        True               78
DS      True               39
Name: count, dtype: int64


Parse spans column

In [4]:
def parse_spans(raw, has_skill):
    # if has_skill is False, treat as no spans even if text exists
    if not has_skill:
        return []
    if pd.isna(raw):
        return []
    text = str(raw).strip()
    if not text:
        return []
    # split on semicolons
    parts = [p.strip() for p in text.split(";")]
    return [p for p in parts if p]

df["gold_spans"] = df.apply(lambda r: parse_spans(r["spans"], r["has_skill_bool"]), axis=1)

Find span matches (case, insensitive, whole phrase)

In [5]:
def find_all_occurrences(sentence, span):
    # Return all (start,end) for span in sentence (case-insensitive).
    s = sentence
    pat = re.escape(span)
    matches = []
    for m in re.finditer(pat, s, flags=re.IGNORECASE):
        matches.append((m.start(), m.end()))
    return matches

def pick_non_overlapping(matches):
    # Given list of (start,end,span_text), keep a non-overlapping set (prefer longer first)
    matches = sorted(matches, key=lambda x: (-(x[1]-x[0]), x[0]))  # longer first, then earlier
    chosen = []
    for st, en, sp in matches:
        overlap = any(not (en <= cst or st >= cen) for cst, cen, _ in chosen)
        if not overlap:
            chosen.append((st, en, sp))
    # return sorted by start
    return sorted(chosen, key=lambda x: x[0])

def get_gold_char_spans(sentence, spans):
    #Try to match each gold span in the sentence; returns matched list + unmatched list.
    all_matches = []
    unmatched = []
    for sp in spans:
        occ = find_all_occurrences(sentence, sp)
        if not occ:
            unmatched.append(sp)
        else:
            # if multiple occurrences, include them all (rare but possible)
            for st, en in occ:
                all_matches.append((st, en, sp))
    # remove overlaps conservatively
    all_matches = pick_non_overlapping(all_matches)
    return all_matches, unmatched

Tokenize + BIO tag

In [6]:
def bio_tag_sentence(sentence, gold_spans):
    doc = nlp(sentence)
    tokens = [t.text for t in doc]
    tags = ["O"] * len(tokens)

    gold_char_spans, unmatched = get_gold_char_spans(sentence, gold_spans)

    # tag tokens by char overlap
    for (st, en, sp_txt) in gold_char_spans:
        # find token indices that overlap [st,en)
        idxs = []
        for i, tok in enumerate(doc):
            if tok.idx < en and (tok.idx + len(tok)) > st:
                idxs.append(i)
        if not idxs:
            continue
        tags[idxs[0]] = "B-SKILL"
        for j in idxs[1:]:
            tags[j] = "I-SKILL"

    return tokens, tags, gold_char_spans, unmatched


records = []
unmatched_rows = []

for i, r in df.iterrows():
    sent = str(r["sentence"])
    gold = r["gold_spans"]

    tokens, tags, matched_char_spans, unmatched = bio_tag_sentence(sent, gold)

    records.append({
        "job_id": r.get("job_id", ""),
        "sent_id": r.get("sent_id", ""),
        "domain": r.get("domain", ""),
        "sentence": sent,
        "tokens": tokens,
        "tags": tags,
        "gold_spans": gold,
        "matched_char_spans": matched_char_spans,
    })

    if unmatched:
        unmatched_rows.append({
            "job_id": r.get("job_id", ""),
            "sent_id": r.get("sent_id", ""),
            "domain": r.get("domain", ""),
            "sentence": sent,
            "gold_spans": gold,
            "unmatched_spans": unmatched,
        })

bio_df = pd.DataFrame(records)
print("BIO rows:", len(bio_df))
print("Rows with any B-SKILL:", (bio_df["tags"].apply(lambda xs: "B-SKILL" in xs)).sum())

unmatched_df = pd.DataFrame(unmatched_rows)
print("Rows with unmatched spans:", len(unmatched_df))

BIO rows: 700
Rows with any B-SKILL: 378
Rows with unmatched spans: 28


Save

In [7]:
BIO_ALL_PATH = OUT_DIR / "bio_all.csv"
bio_df.to_csv(BIO_ALL_PATH, index=False)

UNMATCHED_PATH = OUT_DIR / "bio_unmatched_debug.csv"
unmatched_df.to_csv(UNMATCHED_PATH, index=False)

print("Saved:", BIO_ALL_PATH)
print("Saved:", UNMATCHED_PATH)

Saved: /content/drive/MyDrive/CS685/linkedin/ner_data_v3/bio_all.csv
Saved: /content/drive/MyDrive/CS685/linkedin/ner_data_v3/bio_unmatched_debug.csv


Train/Dev/Test split (stratified by domain)

In [8]:
def stratified_split(df_in, test_frac, seed=42):
    rng = np.random.default_rng(seed)
    test_idxs = []
    for dom, sub in df_in.groupby("domain"):
        n = len(sub)
        k = int(round(n * test_frac))
        k = max(1, k) if n >= 3 else max(0, k)  # avoid weird tiny domains
        pick = rng.choice(sub.index.values, size=k, replace=False)
        test_idxs.extend(pick.tolist())
    test = df_in.loc[test_idxs].copy()
    rest = df_in.drop(index=test_idxs).copy()
    return rest, test

rest, test = stratified_split(bio_df, TEST_FRAC, seed=SEED)
train, dev = stratified_split(rest, DEV_FRAC / (1 - TEST_FRAC), seed=SEED)

In [9]:
# shuffle
train = train.sample(frac=1, random_state=SEED).reset_index(drop=True)
dev   = dev.sample(frac=1, random_state=SEED).reset_index(drop=True)
test  = test.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Split sizes:", len(train), len(dev), len(test))
print("Train domain counts:\n", train["domain"].value_counts())
print("Dev domain counts:\n", dev["domain"].value_counts())
print("Test domain counts:\n", test["domain"].value_counts())

Split sizes: 490 105 105
Train domain counts:
 domain
SWE    322
DA     115
DS      53
Name: count, dtype: int64
Dev domain counts:
 domain
SWE    69
DA     25
DS     11
Name: count, dtype: int64
Test domain counts:
 domain
SWE    69
DA     25
DS     11
Name: count, dtype: int64


In [10]:
TRAIN_PATH = OUT_DIR / "bio_train.csv"
DEV_PATH   = OUT_DIR / "bio_dev.csv"
TEST_PATH  = OUT_DIR / "bio_test.csv"

train.to_csv(TRAIN_PATH, index=False)
dev.to_csv(DEV_PATH, index=False)
test.to_csv(TEST_PATH, index=False)

print("Saved:", TRAIN_PATH)
print("Saved:", DEV_PATH)
print("Saved:", TEST_PATH)

Saved: /content/drive/MyDrive/CS685/linkedin/ner_data_v3/bio_train.csv
Saved: /content/drive/MyDrive/CS685/linkedin/ner_data_v3/bio_dev.csv
Saved: /content/drive/MyDrive/CS685/linkedin/ner_data_v3/bio_test.csv


In [11]:
# Look at one example neatly
row = train.iloc[0]
print("Sentence:", row["sentence"])
print("Tokens:", row["tokens"])
print("Tags:  ", row["tags"])
print("Gold spans:", row["gold_spans"])
print("Matched:", row["matched_char_spans"])

Sentence: 5+ years experience with Cloud technology: GCP, AWS, or Azure.
Tokens: ['5', '+', 'years', 'experience', 'with', 'Cloud', 'technology', ':', 'GCP', ',', 'AWS', ',', 'or', 'Azure', '.']
Tags:   ['O', 'O', 'O', 'O', 'O', 'B-SKILL', 'I-SKILL', 'O', 'B-SKILL', 'O', 'B-SKILL', 'O', 'O', 'B-SKILL', 'O']
Gold spans: ['Cloud technology', 'GCP', 'AWS', 'Azure']
Matched: [(25, 41, 'Cloud technology'), (43, 46, 'GCP'), (48, 51, 'AWS'), (56, 61, 'Azure')]
